In [1]:
!pip install openai pandas numpy python-dotenv mlflow

In [2]:
!pip install openai
!pip install pandas
!pip install numpy
!pip install python-dotenv
!pip install mlflow
!pip install matplotlib
!pip install seaborn
!pip install jupyter
!pip install ipykernel

In [3]:
import openai
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

print("All libraries installed successfully!")

All libraries installed successfully!


In [4]:
!pip install ollama pandas numpy mlflow matplotlib seaborn

In [5]:
from ollama import chat

response = chat(
    model='phi3',
    messages=[
        {
            'role': 'user',
            'content': 'What is Machine Learning?'
        }
    ]
)

print(response['message']['content'])


Machine learning is a subset of artificial intelligence (AI) that provides systems the ability to automatically learn and improve from experience without being explicitly programmed. It focuses on the development of computer programs that can access data, learn from it, and make decisions or predictions. Through the use of algorithms and statistical models, machine learning allows software applications to find hidden insights without being explicitly programmed where to look. It'thy applications range from the simple, like a mail-sorting system that can recognize and sort incoming letters, to complex, like an autonomous vehicle that can navigate the roads without human intervention. Machine learning is being used in various fields such as healthcare, marketing, education, and finance, to improve services, customer experience, and operational efficiency.


In [6]:
import time
import json
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

from ollama import chat

print("Environment Ready")

Environment Ready


In [7]:
questions = [
    "What is Machine Learning?",
    "Explain Supervised Learning.",
    "What is Overfitting?",
    "What is Gradient Descent?",
    "Difference between Classification and Regression?"
]

len(questions)

5

In [8]:
prompt_v1 = """
Answer the following student question clearly.

Question: {question}
"""

In [9]:
results = []

for idx, question in enumerate(questions):

    start_time = time.time()

    response = chat(
        model='phi3',
        messages=[
            {
                'role': 'user',
                'content': prompt_v1.format(
                    question=question
                )
            }
        ]
    )

    end_time = time.time()

    latency = end_time - start_time

    answer = response['message']['content']

    results.append({
    'question_id': idx+1,
    'prompt_version': 'v1',
    'model_name': 'phi3',
    'start_time': start_time,
    'end_time': end_time,
    'latency_seconds': latency,
    'status': 'success',
    'question': question,
    'response': answer
})

df = pd.DataFrame(results)

df

,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status,question,response
0,1,v1,phi3,1.793887e+09,1.793887e+09,13.867775,success,What is Machine Learning?,\nMachine Learning is a subset of artificial i...
1,2,v1,phi3,1.793887e+09,1.793887e+09,31.249595,success,Explain Supervised Learning.,\nSupervised Learning is a type of machine lea...
2,3,v1,phi3,1.793887e+09,1.793887e+09,14.578837,success,What is Overfitting?,Overfitting occurs in machine learning when a ...
3,4,v1,phi3,1.793887e+09,1.793887e+09,18.562186,success,What is Gradient Descent?,\nGradient Descent is an optimization algorith...
4,5,v1,phi3,1.793887e+09,1.793887e+09,26.094000,success,Difference between Classification and Regression?,\nClassification and regression are both types...


In [10]:
print("Mean:", df["latency_seconds"].mean())
print("Median:", df["latency_seconds"].median())
print("Min:", df["latency_seconds"].min())
print("Max:", df["latency_seconds"].max())
print("Std:", df["latency_seconds"].std())

Mean: 20.87047872543335
Median: 18.56218647956848
Min: 13.867774963378906
Max: 31.249595403671265
Std: 7.565746258592798


In [11]:
df.to_csv(
    "latency_tokens_cost.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [12]:
!pip install tiktoken

In [13]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(str(text)))

In [14]:
df["input_tokens"] = df["question"].apply(
    count_tokens
)

df["output_tokens"] = df["response"].apply(
    count_tokens
)

df["total_tokens"] = (
    df["input_tokens"] +
    df["output_tokens"]
)

df[
    [
        "question_id",
        "input_tokens",
        "output_tokens",
        "total_tokens"
    ]
]

,question_id,input_tokens,output_tokens,total_tokens
0,1,5,97,102
1,2,6,219,225
2,3,6,120,126
3,4,6,158,164
4,5,6,185,191


In [15]:
INPUT_PRICE = 0.15
OUTPUT_PRICE = 0.60

In [16]:
df["input_cost"] = (
    df["input_tokens"] / 1_000_000
) * INPUT_PRICE

df["output_cost"] = (
    df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE

df["total_cost"] = (
    df["input_cost"] +
    df["output_cost"]
)

df[
    [
        "question_id",
        "input_tokens",
        "output_tokens",
        "total_cost"
    ]
]

,question_id,input_tokens,output_tokens,total_cost
0,1,5,97,0.000059
1,2,6,219,0.000132
2,3,6,120,0.000073
3,4,6,158,0.000096
4,5,6,185,0.000112


In [17]:
avg_cost = df["total_cost"].mean()

monthly_cost = avg_cost * 10000

print(f"Average Cost Per Request: ${avg_cost:.8f}")
print(f"Projected Monthly Cost for 10,000 Requests: ${monthly_cost:.4f}")

Average Cost Per Request: $0.00009435
Projected Monthly Cost for 10,000 Requests: $0.9435


In [18]:
prompt_v1 = """
Answer the following student question clearly.

Question: {question}
"""


In [19]:
prompt_v2 = """
You are an academic assistant.

Answer the following student question in a clear,
concise and accurate manner.

Use simple language suitable for undergraduate students.

Question: {question}
"""

In [20]:
prompt_v3 = """
You are an academic assistant.

Answer the student question and return ONLY valid JSON.

Do not include markdown or explanations.

{
    "answer":"string",
    "confidence":"high|medium|low",
    "key_points":["string"]
}

Question: {question}
"""

In [21]:
import mlflow

mlflow.set_experiment(
    "eduassist_llmops_assignment"
)

print("Experiment Created")

Experiment Created


In [22]:
import json
import time
import pandas as pd
import mlflow
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(str(text)))

def is_valid_json(text):
    try:
        obj = json.loads(text)

        required_keys = [
            "answer",
            "confidence",
            "key_points"
        ]

        return int(
            all(
                key in obj
                for key in required_keys
            )
        )

    except:
        return 0

# FIXED PROMPTS

prompt_v1 = """
Answer the following student question clearly.

Question: {question}
"""

prompt_v2 = """
You are an academic assistant.

Answer the following student question in a clear,
concise and accurate manner.

Use simple language suitable for undergraduate students.

Question: {question}
"""

prompt_v3 = """
You are an academic assistant.

Answer the student question and return ONLY valid JSON.

Do not include markdown or explanations.

{{
    "answer":"string",
    "confidence":"high|medium|low",
    "key_points":["string"]
}}

Question: {question}
"""

prompts = {
    "v1": prompt_v1,
    "v2": prompt_v2,
    "v3": prompt_v3
}

evaluation_results = []

for version, prompt in prompts.items():

    latencies = []
    input_tokens_list = []
    output_tokens_list = []
    costs = []
    json_scores = []

    print(f"Running {version}...")

    for question in questions:

        start_time = time.time()

        response = chat(
            model="phi3",
            messages=[
                {
                    "role": "user",
                    "content": prompt.format(
                        question=question
                    )
                }
            ]
        )

        end_time = time.time()

        answer = response["message"]["content"]

        latency = end_time - start_time

        input_tokens = count_tokens(question)
        output_tokens = count_tokens(answer)

        cost = (
            (input_tokens / 1_000_000) * INPUT_PRICE +
            (output_tokens / 1_000_000) * OUTPUT_PRICE
        )

        json_score = (
            is_valid_json(answer)
            if version == "v3"
            else 0
        )

        latencies.append(latency)
        input_tokens_list.append(input_tokens)
        output_tokens_list.append(output_tokens)
        costs.append(cost)
        json_scores.append(json_score)

    avg_latency = sum(latencies) / len(latencies)

    avg_input_tokens = (
        sum(input_tokens_list) /
        len(input_tokens_list)
    )

    avg_output_tokens = (
        sum(output_tokens_list) /
        len(output_tokens_list)
    )

    avg_cost = (
        sum(costs) /
        len(costs)
    )

    valid_json_rate = (
        sum(json_scores) /
        len(json_scores)
    )

    with mlflow.start_run(
        run_name=version
    ):

        mlflow.log_metric(
            "avg_latency_seconds",
            avg_latency
        )

        mlflow.log_metric(
            "avg_input_tokens",
            avg_input_tokens
        )

        mlflow.log_metric(
            "avg_output_tokens",
            avg_output_tokens
        )

        mlflow.log_metric(
            "avg_cost_per_request",
            avg_cost
        )

        mlflow.log_metric(
            "valid_json",
            valid_json_rate
        )

        mlflow.set_tag(
            "prompt_version",
            version
        )

    evaluation_results.append({
        "Prompt Version": version,
        "Avg Latency": round(avg_latency, 2),
        "Avg Input Tokens": round(avg_input_tokens, 2),
        "Avg Output Tokens": round(avg_output_tokens, 2),
        "Avg Cost": round(avg_cost, 8),
        "Valid JSON Rate": round(valid_json_rate, 2)
    })

comparison_df = pd.DataFrame(
    evaluation_results
)

comparison_df

Running v1...
Running v2...
Running v3...


,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate
0,v1,22.90,5.8,160.2,0.000097,0.0
1,v2,20.48,5.8,135.8,0.000082,0.0
2,v3,63.89,5.8,289.8,0.000175,0.0


In [23]:
comparison_df.to_csv(
    "prompt_evaluation_results.csv",
    index=False
)

comparison_df

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate
0,v1,22.90,5.8,160.2,0.000097,0.0
1,v2,20.48,5.8,135.8,0.000082,0.0
2,v3,63.89,5.8,289.8,0.000175,0.0


In [24]:
comparison_df.to_csv(
    "prompt_evaluation_results.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [25]:
!mlflow ui

^C


In [27]:
import mlflow

runs = mlflow.search_runs(
    experiment_names=["eduassist_llmops_assignment"]
)

runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.avg_cost_per_request,metrics.avg_input_tokens,metrics.avg_output_tokens,metrics.valid_json,metrics.avg_latency_seconds,tags.mlflow.runName,tags.mlflow.source.type,tags.prompt_version,tags.mlflow.user,tags.mlflow.source.name
0,3e11801bc460434db16d065e47eb19b5,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 14:09:37.191000+00:00,2026-11-05 14:09:37.448000+00:00,0.000175,5.8,289.8,0.0,63.889475,v3,NOTEBOOK,v3,Meghana,mops_assignment2.ipynb
1,0250fc510b734c4692532df83a28156d,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 14:04:17.130000+00:00,2026-11-05 14:04:17.350000+00:00,0.000082,5.8,135.8,0.0,20.476060,v2,NOTEBOOK,v2,Meghana,mops_assignment2.ipynb
2,4f680541d1bd45c28c2c43631da09f5b,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 14:02:34.394000+00:00,2026-11-05 14:02:34.660000+00:00,0.000097,5.8,160.2,0.0,22.903323,v1,NOTEBOOK,v1,Meghana,mops_assignment2.ipynb
3,db45009c1f624304a2df5ab3e1cb9ba3,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 09:04:37.170000+00:00,2026-11-05 09:04:37.575000+00:00,0.000086,5.8,142.0,0.0,23.795890,v3,NOTEBOOK,v3,Meghana,mops_assignment2.ipynb
4,b436dd11533143b9809d07d2af0c1fa3,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 09:02:37.633000+00:00,2026-11-05 09:02:37.968000+00:00,0.000078,5.8,127.8,0.0,20.784761,v2,NOTEBOOK,v2,Meghana,mops_assignment2.ipynb
5,f5d66f977c5d4f3db39e151721e750e7,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 09:00:52.497000+00:00,2026-11-05 09:00:53.451000+00:00,0.000109,5.8,180.2,0.0,29.379280,v1,NOTEBOOK,v1,Meghana,mops_assignment2.ipynb
6,128cd3b8dc2b430c81b8e58d87121d37,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 08:18:45.363000+00:00,2026-11-05 08:18:45.708000+00:00,0.000055,5.8,90.4,0.0,17.033177,v3,NOTEBOOK,v3,Meghana,mops_assignment2.ipynb
7,844cc870bbf84f7b9afbc263f621c772,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 08:17:19.062000+00:00,2026-11-05 08:17:19.305000+00:00,0.000071,5.8,116.2,0.0,17.734102,v2,NOTEBOOK,v2,Meghana,mops_assignment2.ipynb
8,efa4d7eb9d3047eaa98d4e2e55224b70,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 08:15:49.547000+00:00,2026-11-05 08:15:50.245000+00:00,0.000114,5.8,187.8,0.0,30.612904,v1,NOTEBOOK,v1,Meghana,mops_assignment2.ipynb
9,2b6bc6a25f904c018b3f087560e3cf58,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 08:10:31.261000+00:00,2026-11-05 08:10:31.527000+00:00,0.000098,5.8,161.2,0.0,23.596944,v2,NOTEBOOK,v2,Meghana,mops_assignment2.ipynb


In [28]:
runs[[
    "tags.mlflow.runName",
    "metrics.avg_latency_seconds",
    "metrics.avg_input_tokens",
    "metrics.avg_output_tokens",
    "metrics.avg_cost_per_request",
    "metrics.valid_json"
]]

,tags.mlflow.runName,metrics.avg_latency_seconds,metrics.avg_input_tokens,metrics.avg_output_tokens,metrics.avg_cost_per_request,metrics.valid_json
0,v3,63.889475,5.8,289.8,0.000175,0.0
1,v2,20.476060,5.8,135.8,0.000082,0.0
2,v1,22.903323,5.8,160.2,0.000097,0.0
3,v3,23.795890,5.8,142.0,0.000086,0.0
4,v2,20.784761,5.8,127.8,0.000078,0.0
5,v1,29.379280,5.8,180.2,0.000109,0.0
6,v3,17.033177,5.8,90.4,0.000055,0.0
7,v2,17.734102,5.8,116.2,0.000071,0.0
8,v1,30.612904,5.8,187.8,0.000114,0.0
9,v2,23.596944,5.8,161.2,0.000098,0.0
